In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

df = pd.read_csv("mobile_price.csv")

df = df[df["price_range"] == 1].copy()

features = ["ram", "int_memory", "px_width", "battery_power"]
df = df[features]

In [2]:
def categorize_column(col):
    min_v = col.min()
    max_v = col.max()
    r = max_v - min_v

    low_th = min_v + 0.3 * r
    high_th = min_v + 0.7 * r

    def label(x):
        if x <= low_th:
            return "low"
        elif x <= high_th:
            return "medium"
        else:
            return "high"

    return col.apply(label)

# 套用
cat_df = pd.DataFrame()
for col in features:
    cat_df[col] = categorize_column(df[col])

In [3]:
transactions = []

for _, row in cat_df.iterrows():
    transaction = []
    for col in features:
        transaction.append(f"{col}_{row[col]}")
    transactions.append(transaction)

transactions[0]

['ram_high', 'int_memory_low', 'px_width_low', 'battery_power_low']

In [4]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_encoded = pd.DataFrame(te_array, columns=te.columns_)

In [12]:
from mlxtend.frequent_patterns import fpgrowth
warnings.filterwarnings("ignore", category=DeprecationWarning)

frequent_itemsets = fpgrowth(
    df_encoded,
    min_support=0.3,
    use_colnames=True
)

frequent_itemsets = frequent_itemsets.sort_values(by="support", ascending=False)

print("Frequent Patterns (support ≥ 0.3):")
print(frequent_itemsets)

Frequent Patterns (support ≥ 0.3):
   support                            itemsets
2    0.682                        (ram_medium)
3    0.416                   (px_width_medium)
5    0.414              (battery_power_medium)
4    0.412                 (int_memory_medium)
7    0.318  (battery_power_medium, ram_medium)
0    0.316                    (int_memory_low)
1    0.308                 (battery_power_low)
6    0.306       (ram_medium, px_width_medium)


In [13]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.4
)

# 再篩條件
rules = rules[
    (rules["support"] >= 0.3) &
    (rules["confidence"] >= 0.4) &
    (rules["lift"] >= 0.8)
]

rules = rules.sort_values(by="confidence", ascending=False)

print("Association Rules:")
print(rules[["antecedents", "consequents", "support", "confidence", "lift"]])

Association Rules:
              antecedents             consequents  support  confidence  \
0  (battery_power_medium)            (ram_medium)    0.318    0.768116   
3       (px_width_medium)            (ram_medium)    0.306    0.735577   
1            (ram_medium)  (battery_power_medium)    0.318    0.466276   
2            (ram_medium)       (px_width_medium)    0.306    0.448680   

       lift  
0  1.126270  
3  1.078559  
1  1.126270  
2  1.078559  


In [14]:
def format_rule(x):
    return ', '.join(list(x))

rules["antecedents"] = rules["antecedents"].apply(format_rule)
rules["consequents"] = rules["consequents"].apply(format_rule)

print(rules[["antecedents", "consequents", "support", "confidence", "lift"]].round(4))

            antecedents           consequents  support  confidence    lift
0  battery_power_medium            ram_medium    0.318      0.7681  1.1263
3       px_width_medium            ram_medium    0.306      0.7356  1.0786
1            ram_medium  battery_power_medium    0.318      0.4663  1.1263
2            ram_medium       px_width_medium    0.306      0.4487  1.0786
